# 00 — Environment Setup
This notebook inspects the active environment and reports what is available for the project.


## Objectives
- Report Python/interpreter/project path details.
- Check PyTorch + CUDA + GPU visibility.
- Check PyTorch Geometric, pandas, NetworkX.
- Detect SUMO availability.
- Provide remediation hints for missing packages.


In [1]:
import os, sys, platform, subprocess
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())


Project root: /home/runner/work/Q-Optima/Q-Optima/gat-traffic-prediction
Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Executable: /usr/bin/python
Platform: Linux-6.17.0-1022-azure-x86_64-with-glibc2.39


In [2]:
def safe_import(name):
    try:
        module = __import__(name)
        return True, getattr(module, "__version__", "unknown")
    except Exception as e:
        return False, str(e)

for pkg in ["torch", "torch_geometric", "pandas", "networkx", "yaml"]:
    ok, info = safe_import(pkg)
    print(f"{pkg:16} installed={ok} info={info}")


torch            installed=True info=2.14.1+cu130
torch_geometric  installed=False info=No module named 'torch_geometric'
pandas           installed=True info=3.0.6


networkx         installed=True info=3.7
yaml             installed=True info=6.0.1


In [3]:
try:
    import torch
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0, 2.0, 3.0])
    print("Tensor sanity:", (x * 2).tolist())
except Exception as e:
    print("Torch checks failed:", e)


Torch version: 2.14.1+cu130
CUDA available: False
Tensor sanity: [2.0, 4.0, 6.0]


In [4]:
def check_command(cmd):
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return True, out.strip().splitlines()[0]
    except Exception as e:
        return False, str(e)

for cmd in (["sumo", "--version"], ["sumo-gui", "--version"], ["jupyter", "--version"]):
    ok, info = check_command(cmd)
    print(cmd[0], "available=" , ok, "|", info)


sumo available= False | [Errno 2] No such file or directory: 'sumo'
sumo-gui available= False | [Errno 2] No such file or directory: 'sumo-gui'


jupyter available= True | Selected Jupyter core packages...


## Missing dependency guidance
If `torch_geometric` is missing, install with:
- `uv pip install torch-geometric`

If SUMO is missing:
- Keep using synthetic path notebooks (`03`, `08`, `09`, `11`)
- Use `docs/sumo_setup.md` when ready.
